# PSF aberration analysis (lateral FOV + optional depth)

Consumes the results of `psf_extraction.ipynb` (`aligned_stack.npz`,
`detections.csv`, `ALL_psf_avg.tif.json`) and maps PSF metrics over the field of
view via kernel-weighted local PSFs (`psf_utils.local_psf`): coherent averaging
of the aligned bead crops around each grid position, with the Kish effective
sample size (ESS) as the noise flag.

- Lateral maps over (y, x) are always produced.
- Depth (z) dependence is optional: `do_depth='auto'` measures it only if the
  kept beads span more than `depth_min_span_nm` in z - a single bead layer on
  the coverslip has no meaningful depth dependence.

Outputs in `psf_dir/<out_subdir>/`: `local_psf_metrics.csv`, `maps.png`
(and `depth.png` if enabled), `recipe_parameters.json`.

In [ ]:
# extraction results folder (from psf_extraction.ipynb), containing
# aligned_stack.npz, detections.csv, ALL_psf_avg.tif.json
psf_dir = '/data/beads_dataset/psf'
out_subdir = 'aberrations'

# kernel width for local PSFs, in nm (rule of thumb: a few PSF FWHM)
bandwidth_nm = 2000.0
# grid step for local PSF positions, in nm
grid_step_nm = 1000.0
# skip local PSFs with fewer effective samples than this
min_ess = 3.0

# depth dependence: True / False / 'auto'
# 'auto' = measure depth only if kept beads span more than depth_min_span_nm in z
do_depth = 'auto'
depth_min_span_nm = 3000.0

In [ ]:
# << auto-managed by scripts/inject_parameter_logging.py - do not edit >>
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'recipe_params.py').is_file():
        sys.path.insert(0, str(_p))
        break
import recipe_params
recipe_params.capture(globals(), notebook='restoration/psf_aberrations.ipynb',
                     base_dir_key='psf_dir')

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from calmutils.restoration import psf_metrics

for _p in [Path.cwd(), *Path.cwd().parents]:  # sibling module psf_utils.py
    if (_p / 'restoration' / 'psf_utils.py').is_file():
        sys.path.insert(0, str(_p / 'restoration'))
        break
from psf_utils import local_psf, per_bead_metrics

psf_dir = Path(psf_dir)
out_path = psf_dir / out_subdir
out_path.mkdir(parents=True, exist_ok=True)

d = np.load(psf_dir / 'aligned_stack.npz')
stack, used, keep = d['stack'], d['used'], d['keep']
det = pd.read_csv(psf_dir / 'detections.csv')
psz = np.asarray(json.load(open(psf_dir / 'ALL_psf_avg.tif.json'))['psz_nm'], dtype=float)
locs = det[['pos_z', 'pos_y', 'pos_x']].to_numpy()

print(f'psf_dir: {psf_dir}')
print(f'{int(keep.sum())} kept beads of {len(keep)} | psz_nm {psz} | stack {stack.shape}')
print(f'output: {out_path}')

In [ ]:
# per-bead metrics table (from the aligned stack) + depth decision
m = per_bead_metrics(stack, used, psz=psz)
beads = det.copy()
for ax, dname in enumerate('zyx'):
    beads[f'fwhm_{dname}_nm'] = m['fwhm_phys'][:, ax]
    beads[f'offset_{dname}_nm'] = m['centroid_offset_px'][:, ax] * psz[ax]
    beads[f'asym_{dname}'] = m['centroid_offset_px'][:, ax] * 0 + np.nan  # filled below
# asymmetry needs psf_metrics-style marginals per bead; use per-bead profiles:
from calmutils.restoration.psf_metrics import axis_profile, profile_metrics
asym = np.full((len(beads), 3), np.nan)
for i in np.where(used)[0]:
    from psf_utils import _apply_background
    crop = _apply_background(stack[i], 'shell')
    for ax in range(3):
        asym[i, ax] = profile_metrics(axis_profile(crop, ax))['asymmetry']
for ax, dname in enumerate('zyx'):
    beads[f'asym_{dname}'] = asym[:, ax]
beads['fwhm_xy_nm'] = beads[['fwhm_y_nm', 'fwhm_x_nm']].mean(axis=1)
beads['offset_xy_nm'] = np.sqrt(beads['offset_y_nm']**2 + beads['offset_x_nm']**2)

z_span_nm = float((locs[keep][:, 0].max() - locs[keep][:, 0].min()) * psz[0])
if do_depth == 'auto':
    do_depth = bool(z_span_nm > depth_min_span_nm)
print(f'z span of kept beads: {z_span_nm:.0f} nm > depth_min_span_nm {depth_min_span_nm:.0f}'
      f' -> do_depth = {do_depth}')

# overview scatter of per-bead metrics vs position
bk = beads[keep]
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(bk['pos_y'] * psz[1], bk['pos_x'] * psz[2], c=bk['fwhm_xy_nm'], s=40)
axs[0].set_xlabel('y (nm)'); axs[0].set_ylabel('x (nm)')
axs[0].set_title('per-bead FWHM xy (nm) over FOV')
cb = axs[0].collections[0]; fig.colorbar(cb, ax=axs[0])
if do_depth:
    axs[1].scatter(bk['pos_z'] * psz[0], bk['fwhm_xy_nm'], s=30)
    axs[1].set_xlabel('z (nm)'); axs[1].set_title('per-bead FWHM xy vs depth')
else:
    axs[1].scatter(bk['pos_y'] * psz[1], bk['fwhm_xy_nm'], s=30)
    axs[1].set_xlabel('y (nm)'); axs[1].set_title('per-bead FWHM xy vs y (no depth mode)')
fig.tight_layout()

In [ ]:
# local PSFs on a (y, x) grid [x z levels if do_depth], metrics per grid point
step_px = np.asarray(grid_step_nm) / psz
z_levels = ([np.median(locs[keep][:, 0])] if not do_depth else
            list(np.arange(locs[keep][:, 0].min(), locs[keep][:, 0].max() + 1e-9, step_px[0])))
yg = np.arange(locs[keep][:, 1].min(), locs[keep][:, 1].max() + 1e-9, step_px[1])
xg = np.arange(locs[keep][:, 2].min(), locs[keep][:, 2].max() + 1e-9, step_px[2])

records = []
for gz in z_levels:
    for gy in yg:
        for gx in xg:
            psf_l, ess = local_psf(stack, used, locs, psz, [gz, gy, gx],
                                   bandwidth=bandwidth_nm)
            if ess < min_ess:
                continue
            mm = psf_metrics(psf_l, psz)
            records.append({
                'z': gz, 'y': gy, 'x': gx, 'ess': ess,
                'fwhm_z_nm': mm['fwhm_phys'][0],
                'fwhm_xy_nm': float(np.mean(mm['fwhm_phys'][1:])),
                'offset_xy_nm': float(np.linalg.norm(mm['peak_offset_phys'][1:])),
                'asym_xy': float(np.mean(np.abs(mm['asymmetry_px'][1:]))),
                'astigmatism': mm['astigmatism'],
                'sidelobe_ratio': mm['sidelobe_ratio'],
                'edge_level': mm['edge_level'],
            })
loc = pd.DataFrame(records)
print(f'{len(loc)} local PSFs (ess >= {min_ess}); '
      f'z levels: {len(z_levels)}')
loc.head()

In [ ]:
# lateral maps at the z level nearest the median bead depth
z_med = float(np.median(locs[keep][:, 0]))
lat = loc.iloc[(loc['z'] - z_med).abs().argsort()[:len(loc)]]
lat = loc[np.isclose(loc['z'], loc['z'].iloc[(loc['z'] - z_med).abs().argmin()])]

metrics = [('fwhm_xy_nm', 'FWHM xy (nm)'), ('offset_xy_nm', 'peak offset xy (nm)'),
           ('asym_xy', 'mean |asymmetry| xy'), ('astigmatism', 'astigmatism (x-y FWHM)')]
fig, axs = plt.subplots(2, 2, figsize=(11, 9))
for ax, (col, title) in zip(axs.ravel(), metrics):
    pv = lat.pivot_table(index='y', columns='x', values=col)
    im = ax.imshow(pv.values, origin='lower', cmap='viridis',
                   extent=[pv.columns.min() * psz[2], pv.columns.max() * psz[2],
                           pv.index.min() * psz[1], pv.index.max() * psz[1]])
    ax.set_title(f'{title} (z={lat["z"].iloc[0] * psz[0]:.0f} nm, ess>={min_ess:.0f})')
    ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)')
    fig.colorbar(im, ax=ax)
fig.suptitle(f'local PSF maps, bandwidth {bandwidth_nm:.0f} nm, step {grid_step_nm:.0f} nm')
fig.tight_layout()
fig.savefig(out_path / 'maps.png', dpi=150)
plt.show()

if do_depth:
    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (col, title) in zip(axs, metrics):
        grp = loc.groupby('z')[col].mean()
        ax.scatter(loc['z'] * psz[0], loc[col], s=10, alpha=0.4)
        ax.plot(grp.index * psz[0], grp.values, 'r-', lw=2)
        ax.set_xlabel('z (nm)'); ax.set_title(title)
    fig.suptitle('depth dependence of local PSF metrics (red = mean per z level)')
    fig.tight_layout()
    fig.savefig(out_path / 'depth.png', dpi=150)
    plt.show()
else:
    print('depth mode off (single bead layer) - no depth.png')

In [ ]:
loc.to_csv(out_path / 'local_psf_metrics.csv', index=False)
beads[keep].to_csv(out_path / 'per_bead_metrics.csv', index=False)
print('saved to', out_path)
for p in sorted(out_path.iterdir()):
    print('  ', p.name)